# Qutritium Tutorial: Process Tomography

State tomography reconstructs a *state*; process tomography reconstructs a
*channel* — what a gate actually does to every input, noise and all. This is
what v1.5 adds:

1. Building the process-tomography circuits for a single-qutrit gate
2. Reconstructing the channel's **Choi matrix** from measurement counts
3. Reading **Kraus operators** off the Choi matrix
4. Doing it for a noisy gate and checking the result

The recipe: prepare an informationally complete set of inputs (the 12 MUB
states), push each through the gate, and do state tomography on every output.

In [1]:
import numpy as np

import qutritium
from qutritium import DensityMatrixSimulator
from qutritium.channels import depolarizing_channel, NoiseModel
from qutritium.gates import X01
from qutritium.tomography import (choi_to_kraus, process_tomography_circuits, reconstruct_process)

print(f"Qutritium version: {qutritium.__version__}")

Qutritium version: 1.5.0


## A small helper

`process_tomography_circuits(gate)` hands back `(circuits, input_states)`: 12
groups of 4 measurement circuits, and the input density matrix each group
prepares. We run every circuit on the density-matrix simulator (optionally with
a noise model), collect the counts in order, and feed them to
`reconstruct_process`. We use exact probabilities here so the reconstruction is
deterministic — swap in `StatevectorSimulator` + `run()` for sampled counts.

In [2]:
def run_pt(gate, noise_model=None, shots=100_000):
    groups, inputs = process_tomography_circuits(gate)
    counts = []
    for group in groups:
        per_basis = []
        for circ in group:
            dm = DensityMatrixSimulator(circ)
            if noise_model is not None:
                dm.set_noise_model(noise_model)
            probs = dm.probabilities()
            per_basis.append({str(k): round(float(p) * shots) for k, p in enumerate(probs)})
        counts.append(per_basis)
    return reconstruct_process(counts, inputs)

## 1. An ideal gate

Run process tomography on a noiseless `X01`. The Choi matrix is $9\times 9$ and
Hermitian; for a trace-preserving channel its trace is the qutrit dimension, 3.

In [3]:
choi = run_pt(X01())
print("Choi shape:", choi.shape, "| trace:", round(choi.trace().real, 4))

Choi shape: (9, 9) | trace: 3.0


## 2. Kraus operators

`choi_to_kraus` eigendecomposes the Choi matrix and returns one Kraus operator
per eigenvalue above `atol`. A noiseless gate is a rank-1 channel: a single
Kraus operator, equal to the gate's unitary up to a global phase.

In [4]:
kraus = choi_to_kraus(choi, atol=1e-6)
print("number of Kraus operators:", len(kraus))
print("matches X01 (up to global phase):",
      np.allclose(np.abs(kraus[0]), np.abs(X01().matrix()), atol=1e-2))

number of Kraus operators: 1
matches X01 (up to global phase): True


## 3. A noisy gate

Attach a 20% depolarizing channel to `X01` and reconstruct again. Now the channel
is mixed, so it takes several Kraus operators. Two sanity checks: the operators
are trace preserving ($\sum_k K_k^\dagger K_k = I$), and applying the
reconstructed channel to $|0\rangle\langle 0|$ reproduces the expected
$0.8\,(X_{01}|0\rangle\langle 0|X_{01}^\dagger) + 0.2\,I/3$.

In [5]:
nm = NoiseModel()
nm.add_quantum_error(depolarizing_channel(0.2), "X01")  # 20% depolarizing after X01

choi_noisy = run_pt(X01(), nm)
kraus_noisy = choi_to_kraus(choi_noisy, atol=1e-6)
print("number of Kraus operators:", len(kraus_noisy))

total = sum(k.conj().T @ k for k in kraus_noisy)
print("trace preserving (sum K.dag K = I):", np.allclose(total, np.eye(3), atol=1e-2))

rho_in = np.diag([1.0, 0.0, 0.0]).astype(complex)  # |0><0|
rho_out = sum(k @ rho_in @ k.conj().T for k in kraus_noisy)
x = X01().matrix()
expected = 0.8 * (x @ rho_in @ x.conj().T) + 0.2 * np.eye(3) / 3
print("channel action matches expected:", np.allclose(rho_out, expected, atol=1e-2))
print("reconstructed output populations:", np.round(np.diag(rho_out).real, 3))

number of Kraus operators: 9
trace preserving (sum K.dag K = I): True
channel action matches expected: True
reconstructed output populations: [0.067 0.867 0.067]


## Summary

- `process_tomography_circuits(gate)` -> run on a `DensityMatrixSimulator` ->
  `reconstruct_process` gives the $9\times 9$ **Choi matrix** of the gate.
- `choi_to_kraus` turns that into **Kraus operators**: one for a unitary, several
  for a noisy channel.
- A depolarizing `X01` comes back trace preserving and reproduces the expected
  mixed-state output — the channel, noise included, recovered end to end.

The **Tomography** API page has the math and the full reference list.